# Tiny Educational U-Net for 64x64 Images

A small U-Net (Ronneberger et al., 2015) built from plain `torch.nn`
layers only (no external U-Net/model library). This is the first
skip-connection architecture in the thesis path (CNN -> plain
autoencoder -> **U-Net** -> DDPM U-Net), so the goal is to make the
skip-connection / concatenation bookkeeping explicit and correct
before the same skeleton gets reused as a DDPM noise-predictor.

**Architecture** (`TinyUNet64`, defined in
`code/unet_tiny_denoising_64.py`), shape trace for input `[B,3,64,64]`:

```
input                                 [B,  3, 64, 64]
--- encoder ---
enc1 (3->32)  + relu              -> [B, 32, 64, 64]   = skip1
pool1 (2x2 maxpool)                -> [B, 32, 32, 32]
enc2 (32->64) + relu              -> [B, 64, 32, 32]   = skip2
pool2 (2x2 maxpool)                -> [B, 64, 16, 16]
enc3 (64->128)+ relu              -> [B,128, 16, 16]   = skip3
pool3 (2x2 maxpool)                -> [B,128,  8,  8]
--- bottleneck ---
bottleneck (128->256) + relu       -> [B,256,  8,  8]
--- decoder ---
up3 (ConvTranspose 256->128,k2,s2) -> [B,128, 16, 16]
cat(up3, skip3)                    -> [B,256, 16, 16]
dec3 (256->128) + relu             -> [B,128, 16, 16]
up2 (ConvTranspose 128->64,k2,s2)  -> [B, 64, 32, 32]
cat(up2, skip2)                    -> [B,128, 32, 32]
dec2 (128->64) + relu              -> [B, 64, 32, 32]
up1 (ConvTranspose 64->32,k2,s2)   -> [B, 32, 64, 64]
cat(up1, skip1)                    -> [B, 64, 64, 64]
dec1 (64->32) + relu               -> [B, 32, 64, 64]
out_conv (32->3, k1) + sigmoid     -> [B,  3, 64, 64]
```

2x2 maxpool exactly halves H,W at every stage (64->32->16->8, no
rounding), and each `ConvTranspose2d` (kernel=stride=2, padding=0)
gives `out = (in-1)*2 + 2 = 2*in` exactly, so every upsampled map
matches its skip tensor's spatial size and `torch.cat` never needs
cropping. See the class docstring in the `.py` file for the full
derivation.

**Task**: image denoising. A clean image `x` is corrupted with
additive Gaussian noise to make `x_noisy`, and the U-Net learns
`x_noisy -> x_hat ~= x` via `L = ||x - UNet(x_noisy)||^2`. This mirrors
the input/output-image-shape setup a DDPM noise-predictor U-Net uses
(without timestep conditioning or a noise schedule), making it a
natural stepping stone toward that part of the thesis.

Core code lives in `code/unet_tiny_denoising_64.py` (version-
controlled); this notebook is a thin orchestration layer: GPU check,
repo pull, build model + print full shape trace (every skip connection
and concatenation), download real CIFAR-10 resized to 64x64 (cached in
Google Drive), train, plot the loss curve, and show clean/noisy/
denoised qualitative + quantitative results as the required results-
visualization step.

## 1. GPU check

In [ ]:
import torch

# Never hard-code CUDA: fall back to CPU if no GPU is available.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device = {device}")
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))


## 2. Clone/pull repository

Model code lives in `code/unet_tiny_denoising_64.py`, not duplicated in
this notebook. Replace `REPO_URL` with this repository's URL.

In [ ]:
import os

REPO_URL = "https://github.com/RoeiZ/from_zero_to_hero.git"
REPO_DIR = "/content/from_zero_to_hero"

if not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull

import sys
sys.path.insert(0, os.path.join(REPO_DIR, "code"))


## 3. Build model, print full shape trace

One verbose forward pass on a small synthetic batch, purely to print
the `[B,C,H,W]` shape before and after every op -- including, per
level, the shape of each skip tensor at the moment it is saved, the
shapes immediately *before* each `torch.cat` (upsampled map + skip),
and the fused shape immediately *after* it. Not part of training.

In [ ]:
from unet_tiny_denoising_64 import TinyUNet64, count_trainable_parameters

torch.manual_seed(0)

BASE_CHANNELS = 32

model = TinyUNet64(in_channels=3, out_channels=3, base_channels=BASE_CHANNELS).to(device)
print(f"trainable parameters: {count_trainable_parameters(model):,}")

sample_batch = torch.rand(4, 3, 64, 64, device=device)  # [B,3,64,64], synthetic, in [0,1]
_ = model(sample_batch, verbose=True)


## 4. Download real CIFAR-10 (train/test split), resized to 64x64, cached in Drive

CIFAR-10 is natively 32x32; `get_cifar10_denoising_dataloaders` resizes
every image up to 64x64 so it matches the resolution `TinyUNet64` was
designed for. Labels are unused (the denoising target is the clean
image itself) -- this uses `torchvision`'s native train/test partition,
no extra validation split invented.

Colab's local disk (`/content/...`) is wiped whenever the runtime
recycles, so we mount Google Drive and point `DATA_ROOT` at a
persistent Drive folder: `torchvision` downloads into it once and
reads from it on every later run.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

# Persistent cache: downloaded once into Drive, reused on every
# future Colab session (local /content storage is ephemeral).
DATA_ROOT = "/content/drive/MyDrive/thesis_data/cifar10"


In [ ]:
from unet_tiny_denoising_64 import get_cifar10_denoising_dataloaders

BATCH_SIZE = 128
IMAGE_SIZE = 64

train_loader, test_loader = get_cifar10_denoising_dataloaders(DATA_ROOT, BATCH_SIZE, image_size=IMAGE_SIZE)
print(f"train batches = {len(train_loader)}, test batches = {len(test_loader)}")


## 5. Train

Explicit train/eval loops (`train_one_epoch`, `evaluate`): each clean
batch is corrupted with additive Gaussian noise (`noise_std`) and the
U-Net is trained to recover the clean image, MSE loss, Adam
optimizer.

In [ ]:
import torch.nn as nn
from unet_tiny_denoising_64 import train_one_epoch, evaluate

N_EPOCHS = 10
LR = 1e-3
NOISE_STD = 0.3  # std of additive Gaussian noise defining the denoising task

loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

history = {"train_mse": [], "test_mse": []}

for epoch in range(N_EPOCHS):
    train_mse = train_one_epoch(model, train_loader, loss_fn, optimizer, device, NOISE_STD)
    test_mse = evaluate(model, test_loader, loss_fn, device, NOISE_STD)

    history["train_mse"].append(train_mse)
    history["test_mse"].append(test_mse)

    print(f"epoch {epoch + 1:3d}/{N_EPOCHS}  train_mse={train_mse:.4f}  test_mse={test_mse:.4f}")


## 6. Denoising-loss curve (quantitative result over training)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(history["train_mse"], label="train")
ax.plot(history["test_mse"], label="test")
ax.set_xlabel("epoch")
ax.set_ylabel("pixel-wise MSE (denoised vs. clean)")
ax.set_title("Denoising loss")
ax.legend()
fig.tight_layout()
plt.show()

print(f"final test_mse = {history['test_mse'][-1]:.4f}")


## 7. Qualitative + quantitative check: clean vs. noisy vs. denoised

Per-project convention: always look at concrete examples, not only
scalar metrics. Shows 5 real test images as clean (row 1) / noisy
input (row 2) / U-Net output (row 3), with per-image MSE and PSNR
(output vs. clean) printed under each denoised image.

In [ ]:
from unet_tiny_denoising_64 import visualize_denoising

sample_images, _sample_labels = next(iter(test_loader))
sample_images = sample_images.to(device)

visualize_denoising(model, sample_images, NOISE_STD, num_examples=5)


## 8. Test-set average denoising MSE

In [ ]:
final_test_mse = evaluate(model, test_loader, loss_fn, device, NOISE_STD)
print(f"average test-set denoising MSE (noise_std={NOISE_STD}): {final_test_mse:.4f}")
